<a href="https://colab.research.google.com/github/ris2002/Multik30-Eng-De-_RNN_LSTM/blob/main/RNN_LSTM_Translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [66]:
from datasets import load_dataset
import torch
import torch.nn as nn
import torch.optim as optim

dataset = load_dataset("bentrevett/multi30k")
train_data=dataset['train']
test_data=dataset['test']
valid_data=dataset['validation']

In [67]:
len(train_data)

29000

In [68]:
len(test_data)

1000

In [69]:
len(valid_data)

1014

Data Checking

In [70]:
for item in train_data:
  print(item['en'])
  print(item['de'])
  break



Two young, White males are outside near many bushes.
Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.




> Data Cleaning



In [71]:
import re
def clean_data(text,lang):
  text=text.lower().strip()
  if lang=='en':
    text = re.sub(r"[^a-zA-Z\s]", "", text)
  elif lang=='de':
    text=re.sub(r"[^a-zA-ZäöüßÄÖÜ\s]", "", text)
  text = re.sub(r"\s+", " ", text) #cleans extra white space
  return text






In [40]:
cleaned_results = []
for item in train_data:
  en_cleaned=clean_data(item['en'], 'en')
  de_cleaned=clean_data(item['de'],'de')
  cleaned_results.append({'en': en_cleaned, 'de': de_cleaned})


In [41]:
for item in cleaned_results[:5]:
    print(f"EN: {item['en']}")
    print(f"DE: {item['de']}")
    print("-" * 10)

EN: two young white males are outside near many bushes
DE: zwei junge weiße männer sind im freien in der nähe vieler büsche
----------
EN: several men in hard hats are operating a giant pulley system
DE: mehrere männer mit schutzhelmen bedienen ein antriebsradsystem
----------
EN: a little girl climbing into a wooden playhouse
DE: ein kleines mädchen klettert in ein spielhaus aus holz
----------
EN: a man in a blue shirt is standing on a ladder cleaning a window
DE: ein mann in einem blauen hemd steht auf einer leiter und putzt ein fenster
----------
EN: two men are at the stove preparing food
DE: zwei männer stehen am herd und bereiten essen zu
----------


In [42]:
cleaned_results[-1]


{'en': 'a man in shorts and a hawaiian shirt leans over the rail of a pilot boat with fog and mountains in the background',
 'de': 'ein mann in shorts und hawaiihemd lehnt sich über das geländer eines lotsenboots mit nebel und bergen im hintergrund'}

Tokenisation Pipeline


In [43]:
!python -m spacy download de_core_news_sm


  Using cached https://github.com/explosion/spacy-models/releases/download/de_core_news_sm-3.8.0/de_core_news_sm-3.8.0-py3-none-any.whl (14.6 MB)
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [44]:
import spacy
from spacy.lang.en import English
from spacy.lang.de import German
nlp_en=English()
nlp_de=German()
def tokenize_eng(cleaned_results:list,en_tokens:list):

  tokenizer=nlp_en.tokenizer
  for text in cleaned_results:
    token_list=[]
    eng_text=text['en']

    doc=tokenizer(eng_text)
    for token in doc:
      token_list.append(token.text)
    en_tokens.append(token_list)

def tokenize_de(cleaned_results:list,de_tokens:list):

  tokenizer=nlp_de.tokenizer
  for text in cleaned_results:
    token_list=[]
    de_text=text['de']

    doc=tokenizer(de_text)
    for token in doc:
      token_list.append(token.text)
    de_tokens.append(token_list)







In [45]:
en_tokens=[]
de_tokens=[]

In [46]:
tokenize_eng(cleaned_results,en_tokens)
tokenize_de(cleaned_results,de_tokens)

In [47]:
en_tokens[:5]

[['two',
  'young',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes'],
 ['several',
  'men',
  'in',
  'hard',
  'hats',
  'are',
  'operating',
  'a',
  'giant',
  'pulley',
  'system'],
 ['a', 'little', 'girl', 'climbing', 'into', 'a', 'wooden', 'playhouse'],
 ['a',
  'man',
  'in',
  'a',
  'blue',
  'shirt',
  'is',
  'standing',
  'on',
  'a',
  'ladder',
  'cleaning',
  'a',
  'window'],
 ['two', 'men', 'are', 'at', 'the', 'stove', 'preparing', 'food']]

In [48]:
de_tokens[:5]

[['zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche'],
 ['mehrere',
  'männer',
  'mit',
  'schutzhelmen',
  'bedienen',
  'ein',
  'antriebsradsystem'],
 ['ein',
  'kleines',
  'mädchen',
  'klettert',
  'in',
  'ein',
  'spielhaus',
  'aus',
  'holz'],
 ['ein',
  'mann',
  'in',
  'einem',
  'blauen',
  'hemd',
  'steht',
  'auf',
  'einer',
  'leiter',
  'und',
  'putzt',
  'ein',
  'fenster'],
 ['zwei', 'männer', 'stehen', 'am', 'herd', 'und', 'bereiten', 'essen', 'zu']]

Build a vocab dictionary

In [49]:
from collections import Counter
def build_dictionary(tokens,min_freq=2):
  counter=Counter()
  for text in tokens:
    counter.update(text)
  vocab={"<unk>": 0, "<pad>": 1, "<sos>": 2, "<eos>": 3}
  for word,freq in counter.items():
    if freq>=min_freq:
      vocab[word]=len(vocab)# mapping it to the current size of the 'vocab dict'
  itos={}#index_to_string
  for string,index in vocab.items():
    itos[index]=string
  return vocab,itos

In [50]:
en_vocab,en_itos=build_dictionary(en_tokens)
de_vocab,de_itos=build_dictionary(de_tokens)

In [51]:
en_vocab['<unk>']

0

In [52]:
de_vocab['dahinter']

941

In [53]:
en_itos[0]

'<unk>'

In [54]:
de_itos[7]

'männer'

In [55]:
def mapping(vocab,text):
  mapped_lists=[]
  default=vocab['<unk>']     # there should be no text, only numbers  in the code, previously you put like this default="<unk>"
  for word in text:
    x=vocab.get(word,default)
    mapped_lists.append(x)
  return torch.tensor(mapped_lists) #the embedding layer expects tensors and this func doesnt change the diamention


In [56]:
mapped_en_texts=[]
for text in en_tokens:
  en_mapped_lists=mapping(en_vocab,text)
  mapped_en_texts.append(en_mapped_lists)
mapped_de_texts=[]
for text in de_tokens:
  de_mapped_lists=mapping(de_vocab,text)
  mapped_de_texts.append(de_mapped_lists)



In [57]:
mapped_en_texts[:2]

[tensor([ 4,  5,  6,  7,  8,  9, 10, 11, 12]),
 tensor([13, 14, 15, 16, 17,  8, 18, 19, 20, 21, 22])]

In [58]:
len(mapped_de_texts)

29000

In [59]:
mapped_en_texts[:2]

[tensor([ 4,  5,  6,  7,  8,  9, 10, 11, 12]),
 tensor([13, 14, 15, 16, 17,  8, 18, 19, 20, 21, 22])]

In [60]:
mapped_de_texts[5].shape

torch.Size([14])

In [61]:
from torch.nn.utils.rnn import pad_sequence
padded_en_texts=pad_sequence(mapped_en_texts,batch_first=True,padding_value=0)
padded_de_texts=pad_sequence(mapped_de_texts,batch_first=True,padding_value=0)

In [62]:
padded_en_texts.shape

torch.Size([29000, 37])

# **Encoder**

In [63]:
class Vanilla_RNN_Encoder(nn.Module):
  def __init__(self,input_features,input_vocab_size,input_neurons,num_layers,num_dropout):
    super(Vanilla_RNN_Encoder,self).__init__()
    self.dropout=nn.Dropout(num_dropout)
    self.input_embeddings=nn.Embedding(input_vocab_size,input_features)
    self.rnn=nn.RNN(input_features,input_neurons,num_layers=num_layers,dropout=num_dropout,batch_first=True)

  def forward(self,input_tensors,h0=None):
    input_embeddings=self.dropout(self.input_embeddings(input_tensors))
    encoder_out,h_encoder=self.rnn(input_embeddings,h0)
    return h_encoder



**Clarifications regarding Encoder**


1. 'nn.Embedding' only creates a  empty look up table no vals are created, it needs inputs to fill it values, here we have just initialized
2. In  'input_embeddings'  we are trying to convert words into embeddings, each row in this embedding matrix represents each word in the dictionary, row is size of input vocab abd col is richer rep of text(input_embedding_dim)
3. The rnn gives out hidden layer at each steps(a list [h1,h2,...hn]'encoder_out') and the last hidden state(h_encoder)
4. When you use batch_first=True with DataLoader — unsqueeze and squeeze are not needed because DataLoader already adds the batch dimension automatically.




**Internal workings-----**
1. The input tensors should be in a uniform shape. When tensors are created from tokenised lists it would be like [[1,2,3],[16,25,36,43,52],[65,8,98,56,3526]] here we can decode that the rows is the size of the batch and the col are the length of the sequence of the sentence or seq_lenght, therefore thewre shape will be (batchsize x seq_length). this will be padded so that thhe the no. of rows will be uniform.
2. nn.Embedding laayer is the lookup table of size (vocab_size x input_features). For n words a matrix of random numbers are generated (which will be changed during back prop)
ex-

row 0: [0.1,  0.2,  0.3,  0.4]   ← <unk>

row 1: [0.0,  0.0,  0.0,  0.0]   ← <pad>

row 4: [0.3,  0.2, -0.1,  0.5]   ← "I"

row 5: [-0.1, 0.4,  0.3,  0.2]   ← "love"

row 6: [0.2, -0.3,  0.4,  0.1]   ← "dogs

3. When an input tensor is passed through the embedding layer what happens is essentially these rows are feteched
ex-

[[
  
  [0.3,  0.2, -0.1,  0.5],

  [-0.1, 0.4,  0.3,  0.2],

  [0.2, -0.3,  0.4,  0.1] ,

]] so thhe numbers are replaced by the corresponding rows changing its shape to
(batchsize x seq_length x input_features )

4. By default RNN expects input of shape (seq_len x batch x embed_dim) , when batch_first=Ture it accpets inputs of size (batchsize x seq_length x input_features )
5. Inside RNN , it gives out put of shape (batch x seq_len x no_neurons)
6. Data Loader also can be used to pad the data and make it into size (batchsize x seq_length), it can also be used to shuffle data using training. and for h_encoder   = (num_layers x batch x hidden_dim)

7.  In this notebook I have already padded for my reference , but i will still use the dataloader as this is the correct production way.
8. Final logit size after going to Linear layer gives (batch x seq_len x output_vocab_size)




# **Decoder**

In [64]:
class Vanilla_RNN_Decoder(nn.Module):
  def __init__(self,output_features,output_vocab_size,output_neurons,num_layers,num_dropout):
    super(Vanilla_RNN_Decoder,self).__init__()
    self.dropout=nn.Dropout(num_dropout)
    self.output_embeddings=nn.Embedding(output_vocab_size,output_features)
    self.rnn=nn.RNN(output_features,output_neurons,num_layers=num_layers,dropout=num_dropout,batch_first=True)
    self.output_logits=nn.Linear(output_neurons,output_vocab_size)

  def forward(self,output_tensors,h0=None):
    output_embeddings=self.dropout(self.output_embeddings(output_tensors))
    decoder_out, h_decoder=self.rnn(output_embeddings,h0)
    logits=self.output_logits(decoder_out)
    return h_decoder,logits

# ***Seq-Seq Model***

The input to the Seq2Seq model is processed in batches. A batch consists of multiple English and German sentence pairs grouped together. Each sentence is first tokenised and converted to integer indices using the vocabulary dictionary. Since sentences have different lengths, they are padded using pad_sequence to ensure uniform shape of (batch_size x seq_len). This padded tensor is then passed through the embedding layer which replaces each integer index with its corresponding row from the embedding lookup table, changing the shape to (batch_size x seq_len x embed_dim). The encoder processes this full batch simultaneously — producing a context vector which is the final hidden state of shape (num_layers x batch_size x hidden_dim). This context vector is passed to the decoder as the initial hidden state. The decoder then runs in a loop — at each timestep it takes one token from all sentences simultaneously and predicts the next word for every sentence in the batch at once. During training, teacher forcing is used where the actual correct German word is fed as the next input instead of the predicted word, preventing error cascading.



```
batch = 3 sentences

trg = [[<sos>, "ich",  "liebe",  "hunde",  <eos>],   ← sentence 1
       [<sos>, "er",   "läuft",  "schnell", <eos>],  ← sentence 2
       [<sos>, "das",  "ist",    "gut",    <eos>]]    ← sentence 3

shape = (batch=3 x seq_len=5)

trg[0, :] = [<sos>, <sos>, <sos>]   ← one <sos> per sentence
shape = (batch=3,)

t=1 → predict word 1 for all 3 sentences at once
t=2 → predict word 2 for all 3 sentences at once
t=3 → predict word 3 for all 3 sentences at once
```



In [65]:
class Vanilla_RNN_Seq(nn.Module):
  def __init__(self,decoder,encoder,device):
    super(Vanilla_RNN_Seq,self).__init__()
    self.encoder=encoder
    self.decoder=decoder
    self.device=device
    assert(encoder.input_neurons==decoder.output_neurons)
    assert(encoder.num_layers==decoder.num_layers)
  def forward(self,trg,src,teacher_forcing_ratio):
    h_encoder=self.encoder(src)
    trg_batch_size=trg.shape[0]
    trg_seq_len=trg.shape[1]
    outputs=torch.zeros(trg_batch_size,trg_seq_len,self.decoder.output_vocab_size).to(self.device)
    input=trg[:, 0]
    for t in range(1,trg_seq_len):
      h_decoder,decoder_logits=self.decoder(input,h_encoder)
      outputs[t]=decoder_logits
      teacher_force= random.random()<teacher_forcing_ratio
      top1=decoder_logits.argmax(1)
      input = trg[:,t] if teacher_force else top1
    return outputs



